# Mat & Comp – ML 4: Clasificación con k vecinos
## Dime con quién andas y te diré quién eres

**Curso:** Machine learning y redes neuronales (Sección 3) · **Duración sugerida:** 2 sesiones de 50 minutos · **Requisito:** ML 3

### Objetivos de aprendizaje
Al terminar esta lección podrás:
1. Explicar la diferencia entre **regresión** (predecir un número) y **clasificación** (predecir una categoría).
2. Medir qué tan parecidos son dos datos con la **distancia** de Pitágoras, usando numpy.
3. Programar desde cero el algoritmo de los **k vecinos más cercanos** (k-NN) y dibujar su **mapa de decisión**.
4. Ver cómo el valor de **k** y la **escala** de las variables cambian las predicciones.
5. Usar `KNeighborsClassifier` de **scikit-learn** y compararlo con tu versión.

## 🔍 ¿Mandarina o limón?

En una finca del Tolima miden el **ancho** y el **alto** de varias frutas. Llega una fruta nueva que mide **7.0 cm de ancho y 7.1 cm de alto**. Nadie la ha visto. ¿Es una mandarina o un limón?

✏️ *Antes de seguir, piensa: si tuvieras una tabla con las medidas de 40 frutas ya identificadas, ¿cómo decidirías?*

**Mi idea:**

Una idea muy natural es buscar las frutas de la tabla que **más se parecen** a la nueva y ver qué son. Ese es exactamente el algoritmo de hoy. Y veremos que la respuesta depende de **a cuántas** frutas les preguntes.

Hasta ahora (ML 2 y ML 3) predijimos **números**, como la nota. Eso se llama **regresión**. Hoy vamos a predecir **categorías**, como «mandarina» o «limón». Eso se llama **clasificación**.

## 1. Los datos: 40 frutas

Generamos las medidas de 20 mandarinas (más anchas que altas) y 20 limones (más altos que anchos). Usamos una semilla para que todos obtengamos los mismos números.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(4)
n = 20
mandarinas = np.column_stack([rng.normal(7.4, 0.55, n), rng.normal(6.6, 0.5, n)]).round(1)
limones    = np.column_stack([rng.normal(6.0, 0.5, n),  rng.normal(7.9, 0.55, n)]).round(1)

X = np.vstack([mandarinas, limones])                 # 40 filas: [ancho, alto]
y = np.array(["mandarina"] * n + ["limón"] * n)      # la respuesta correcta de cada fila

print("Forma de X:", X.shape)
print("Primeras 3 frutas:", X[:3].tolist(), y[:3])
print("Últimas 3 frutas: ", X[-3:].tolist(), y[-3:])

En machine learning se acostumbra llamar **`X`** a la tabla de **características** (lo que medimos) y **`y`** a las **etiquetas** (lo que queremos predecir). Cada fila de `X` es una fruta.

In [ ]:
nueva = np.array([7.0, 7.1])

plt.figure(figsize=(6, 6))
plt.scatter(mandarinas[:, 0], mandarinas[:, 1], color="orange", label="mandarina", s=50)
plt.scatter(limones[:, 0], limones[:, 1], color="yellowgreen", label="limón", s=50)
plt.scatter(nueva[0], nueva[1], color="black", marker="*", s=300, label="¿nueva?")
plt.xlabel("ancho (cm)")
plt.ylabel("alto (cm)")
plt.title("40 frutas identificadas y una desconocida")
plt.legend()
plt.axis("equal")
plt.grid(alpha=0.3)
plt.show()

✏️ *Mirando la gráfica, ¿qué dirías que es la fruta nueva? ¿Estás muy seguro?*

## 2. ¿Qué tan parecidas son dos frutas? La distancia

Si cada fruta es un punto en el plano, «parecerse» significa «estar cerca». La distancia entre dos puntos se calcula con el **teorema de Pitágoras** (Lección 9 de la Sección 2):

$$d = \sqrt{(\text{ancho}_1 - \text{ancho}_2)^2 + (\text{alto}_1 - \text{alto}_2)^2}$$

In [ ]:
fruta0 = X[0]
d = np.sqrt((fruta0[0] - nueva[0])**2 + (fruta0[1] - nueva[1])**2)
print("Fruta 0:", fruta0, "→ distancia a la nueva:", round(d, 2), "cm")

Con numpy calculamos las **40 distancias a la vez**, sin ciclos (como en Python 11):

- `X - nueva` resta la fruta nueva a **cada fila**;
- `** 2` eleva al cuadrado cada número;
- `.sum(axis=1)` suma cada fila (ancho² + alto²);
- `np.sqrt` saca la raíz de cada resultado.

In [ ]:
distancias = np.sqrt(((X - nueva) ** 2).sum(axis=1))
print(distancias.round(2))

## 3. El vecino más cercano (k = 1)

La regla más simple: la fruta nueva es lo mismo que la fruta **más cercana**. `np.argmin` nos dice la posición de la distancia más pequeña.

In [ ]:
i = np.argmin(distancias)
print("El vecino más cercano es la fruta", i, "→", X[i], y[i], f"(a {distancias[i]:.2f} cm)")

¡El vecino más cercano es un **limón**! Pero en la gráfica la fruta nueva parece rodeada de mandarinas… Miremos los **7 vecinos** más cercanos. `np.argsort` da las posiciones ordenadas de menor a mayor distancia.

In [ ]:
orden = np.argsort(distancias)
for puesto, j in enumerate(orden[:7], start=1):
    print(f"{puesto}. {y[j]:10} {X[j]}  a {distancias[j]:.2f} cm")

El limón más cercano es un caso raro: un limón bajito y ancho. Si solo le preguntamos a **un** vecino, ese caso raro decide. Si le preguntamos a **varios**, la mayoría dice «mandarina».

## 4. Los k vecinos más cercanos

El algoritmo **k-NN** (*k nearest neighbors*) es:

1. Calcular la distancia del punto nuevo a **todos** los datos.
2. Tomar los **k** más cercanos.
3. **Votar**: la categoría que más se repita entre ellos es la predicción.

`np.unique(..., return_counts=True)` cuenta cuántas veces aparece cada etiqueta (como el conteo de dados de Python 11).

In [ ]:
def predecir_knn(X, y, punto, k):
    distancias = np.sqrt(((X - punto) ** 2).sum(axis=1))
    vecinos = np.argsort(distancias)[:k]                  # posiciones de los k más cercanos
    etiquetas, votos = np.unique(y[vecinos], return_counts=True)
    return etiquetas[np.argmax(votos)]                     # la más votada

for k in [1, 3, 5, 7, 15]:
    print(f"k = {k:2}: {predecir_knn(X, y, nueva, k)}")

💡 Con dos categorías conviene usar **k impar**, para que nunca haya empate en la votación.

Eso es todo el algoritmo: no hay fórmula, no hay pendiente, no hay entrenamiento. El modelo **es** la tabla de datos. Por eso se dice que k-NN es un algoritmo «perezoso»: no aprende nada por adelantado y hace todo el trabajo cuando le preguntas.

## 5. El mapa de decisión

¿Qué predice el modelo en **cada** punto del plano? Pintemos una cuadrícula de puntos con el color de su predicción (con `np.meshgrid`, como en Python 11). La línea donde cambia el color se llama **frontera de decisión**.

In [ ]:
def mapa_de_decision(k, eje):
    a = np.linspace(4.5, 9, 120)
    h = np.linspace(5, 9.5, 120)
    A, H = np.meshgrid(a, h)
    Z = np.zeros(A.shape)
    for f in range(A.shape[0]):
        for c in range(A.shape[1]):
            pred = predecir_knn(X, y, np.array([A[f, c], H[f, c]]), k)
            Z[f, c] = 1 if pred == "mandarina" else 0
    eje.contourf(A, H, Z, levels=[-0.5, 0.5, 1.5], colors=["#e8f5c8", "#ffe0b0"])
    eje.scatter(mandarinas[:, 0], mandarinas[:, 1], color="orange", s=25)
    eje.scatter(limones[:, 0], limones[:, 1], color="yellowgreen", s=25)
    eje.scatter(*nueva, color="black", marker="*", s=200)
    eje.set_title(f"k = {k}")
    eje.set_xlabel("ancho (cm)")

fig, ejes = plt.subplots(1, 3, figsize=(15, 5))
for eje, k in zip(ejes, [1, 5, 15]):
    mapa_de_decision(k, eje)
ejes[0].set_ylabel("alto (cm)")
plt.show()

Observa:
- Con **k = 1** la frontera es muy irregular: cada fruta rara crea su propia «isla». El modelo se cree **todo**, incluso los casos raros.
- Con **k = 15** la frontera es suave: los casos raros pierden importancia.
- Si k es **demasiado** grande (por ejemplo k = 40, todos los datos), todos los puntos votan siempre igual y el modelo deja de mirar dónde está el punto nuevo.

Elegir k es buscar un equilibrio. En ML 7 aprenderemos a hacerlo de forma sistemática; ese problema se llama **sobreajuste**.

## 6. Datos reales: ¿aprobará matemáticas?

Volvamos a los 60 estudiantes de ML 1. Ahora queremos **clasificar**: ¿aprobó (nota ≥ 3.0) o no? Usaremos dos características: horas de estudio y horas de sueño. Como en ML 1, quitamos las filas con datos faltantes (`dropna`).

In [ ]:
import pandas as pd

url = "https://raw.githubusercontent.com/emersonjleon/compumath/main/lecciones/seccion3_ml/datos/estudiantes.csv"
datos = pd.read_csv(url).dropna()
datos["aprobo"] = datos["nota_matematicas"] >= 3.0

print(len(datos), "estudiantes con datos completos")
print(datos["aprobo"].value_counts())

In [ ]:
Xe = datos[["horas_estudio", "horas_sueno"]].to_numpy()
ye = np.where(datos["aprobo"], "aprobó", "no aprobó")

plt.figure(figsize=(7, 5))
for etiqueta, color in [("aprobó", "seagreen"), ("no aprobó", "crimson")]:
    sel = ye == etiqueta
    plt.scatter(Xe[sel, 0], Xe[sel, 1], color=color, label=etiqueta, s=50, alpha=0.7)
plt.xlabel("horas de estudio por semana")
plt.ylabel("horas de sueño por noche")
plt.legend()
plt.title("¿Quién aprobó?")
plt.show()

nuevo = np.array([8, 6])          # 8 horas de estudio, 6 de sueño
print("Predicción para un estudiante con 8 h de estudio y 6 h de sueño:", predecir_knn(Xe, ye, nuevo, 5))

### ¿Qué tan bueno es el modelo?

Una idea tentadora: predecir a cada estudiante de la tabla y contar los aciertos. Pero cuidado: con k = 1, el vecino más cercano de cada estudiante **es él mismo** (distancia 0), así que acertaríamos el 100 % ¡haciendo trampa!

La forma honesta es **dejar uno afuera**: para cada estudiante, lo sacamos de la tabla, lo predecimos con los demás y vemos si acertamos. `np.delete` quita una fila.

In [ ]:
def precision_dejando_uno(X, y, k):
    aciertos = 0
    for i in range(len(X)):
        X_otros = np.delete(X, i, axis=0)
        y_otros = np.delete(y, i)
        if predecir_knn(X_otros, y_otros, X[i], k) == y[i]:
            aciertos += 1
    return aciertos / len(X)

ks = [1, 3, 5, 7, 9, 11, 15, 21, 31]
precisiones = [precision_dejando_uno(Xe, ye, k) for k in ks]
for k, p in zip(ks, precisiones):
    print(f"k = {k:2}: {p:.0%} de aciertos")

plt.plot(ks, precisiones, "o-")
plt.xlabel("k")
plt.ylabel("precisión (dejando uno afuera)")
plt.title("¿Qué k funciona mejor?")
plt.grid(alpha=0.3)
plt.show()

Para comparar: si siempre dijéramos «aprobó», acertaríamos 34 de 57, el **60 %**. k-NN mejora mucho esa respuesta ingenua. ✏️ *¿Qué pasa con la precisión cuando k es muy grande? ¿Por qué?*

## 7. ⚠️ La escala importa

¿Qué pasa si alguien mide el sueño en **minutos** en vez de horas? Los datos dicen exactamente lo mismo… pero las distancias cambian: una diferencia de 1 hora de sueño ahora vale 60, y aplasta a las horas de estudio.

In [ ]:
Xe_min = Xe.copy()
Xe_min[:, 1] = Xe_min[:, 1] * 60          # sueño en minutos

for k in [5, 9, 15]:
    print(f"k = {k:2}:  en horas {precision_dejando_uno(Xe, ye, k):.0%}   ·   sueño en minutos {precision_dejando_uno(Xe_min, ye, k):.0%}")

El modelo empeora solo por cambiar las unidades: ahora casi solo «mira» el sueño. La solución es **normalizar** cada columna (como en el ejercicio 5 de ML 3): restarle su promedio y dividir por su desviación estándar, para que todas queden en la misma escala.

In [ ]:
def normalizar(X):
    return (X - X.mean(axis=0)) / X.std(axis=0)

for k in [5, 9, 15]:
    print(f"k = {k:2}:  normalizando (horas) {precision_dejando_uno(normalizar(Xe), ye, k):.0%}   ·   normalizando (minutos) {precision_dejando_uno(normalizar(Xe_min), ye, k):.0%}")

Normalizadas, las dos versiones dan **exactamente** lo mismo: ya no importa en qué unidades se midió. Fíjate que en este caso los datos en horas, sin normalizar, daban un poco más. Fue casualidad: la variable más importante (las horas de estudio) tenía justo la escala más grande. Normalizar no garantiza el mejor resultado, pero evita que **las unidades** decidan por nosotros. **Regla:** en k-NN (y en casi todo el machine learning) normaliza las características antes de medir distancias.

## 8. Con scikit-learn

**scikit-learn** es la librería de machine learning más usada en Python, y ya viene en Colab. Todos sus modelos se usan igual:

1. **crear** el modelo: `modelo = KNeighborsClassifier(n_neighbors=5)`;
2. **entrenar** con `modelo.fit(X, y)` (en k-NN, «entrenar» es solo guardar la tabla);
3. **predecir** con `modelo.predict(...)`.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

modelo = KNeighborsClassifier(n_neighbors=5)
modelo.fit(X, y)

print("Predicción:", modelo.predict([[7.0, 7.1]]))
print("Categorías:", modelo.classes_)
print("Probabilidades:", modelo.predict_proba([[7.0, 7.1]]))

`predict_proba` dice qué fracción de los vecinos votó por cada categoría: aquí 1 de 5 vecinos es limón (0.2) y 4 de 5 son mandarinas (0.8). Es una medida de **qué tan seguro** está el modelo.

Comparemos con nuestra versión en todos los puntos de la cuadrícula:

In [ ]:
a = np.linspace(4.5, 9, 40)
h = np.linspace(5, 9.5, 40)
puntos = np.array([[ai, hi] for ai in a for hi in h])

nuestra = np.array([predecir_knn(X, y, p, 5) for p in puntos])
de_sklearn = modelo.predict(puntos)
print(f"Coinciden en {np.mean(nuestra == de_sklearn):.1%} de {len(puntos)} puntos")

¡Nuestra versión de 5 líneas hace lo mismo que la de scikit-learn! La de la librería es más rápida y tiene más opciones, pero ahora sabes **exactamente** qué hace por dentro.

## 9. Errores comunes

**Error 1: pasar un solo punto como lista simple.** scikit-learn siempre espera una **tabla** (una lista de filas), aunque sea de una sola fila:

In [ ]:
modelo.predict([7.0, 7.1])        # falta un par de corchetes: debería ser [[7.0, 7.1]]

**Error 2: medir la precisión con los mismos datos** y creer que el modelo es perfecto. Con k = 1 siempre da 100 %. Usa «dejar uno afuera» (o, como veremos en ML 7, separar datos de prueba).

**Error 3: olvidar normalizar.** Si una variable tiene números mucho más grandes que las otras, domina las distancias (sección 7).

**Error 4: k par con dos categorías.** Puede haber empates, y entonces la predicción depende de un detalle interno (`np.unique` elige la primera en orden alfabético).

## 🤔 ¿Qué pasa si…?

Usa la celda de abajo. Antes de ejecutar, **predice** el resultado.

1. …predices la fruta nueva con k = 40 (todas las frutas)? ¿Y con k = 41?
2. …predices un punto que está **exactamente** encima de una fruta de la tabla, con k = 1?
3. …predices una fruta gigante, `[20, 20]`? ¿Tiene sentido la respuesta?
4. …usas k = 2 con la fruta nueva? ¿Quién gana el empate y por qué?
5. …usas `KNeighborsClassifier(n_neighbors=5)` con los estudiantes y le pides `predict_proba([[8, 6]])`?

In [ ]:
# Prueba aquí tus ideas

## ✍️ Ejercicios

**Ejercicio 1 (fácil). A mano.** Calcula con lápiz la distancia entre las frutas `[7.4, 6.5]` y `[6.1, 7.9]`. Luego compruébala con numpy.

**Ejercicio 2 (fácil).** Predice qué es una fruta de **6.5 cm de ancho y 7.4 cm de alto** con k = 1, 3, 7 y 15. ¿Cambia la respuesta? Dibuja la fruta sobre el mapa de decisión.

**Ejercicio 3 (medio). Tres categorías.** Agrega 20 **naranjas**, más grandes: `rng.normal(8.5, 0.6, n)` de ancho y `rng.normal(8.3, 0.6, n)` de alto. Rehaz `X` y `y` con las 60 frutas y dibuja el mapa de decisión con k = 5 (usa tres colores; en `Z` pon 0, 1 o 2). ¿Funciona `predecir_knn` sin cambios?

**Ejercicio 4 (medio). Más características.** Repite la sección 6 con **tres** características: horas de estudio, de sueño y de redes. Normalízalas y calcula la precisión dejando uno afuera para k = 3, 5, 9 y 15. ¿Mejora respecto a usar solo dos?

**Ejercicio 5 (medio). k-NN para regresión.** k-NN también puede predecir **números**: en lugar de votar, se **promedian** las notas de los k vecinos. Escribe `predecir_nota(X, notas, punto, k)` usando solo `horas_estudio` y calcula el error cuadrático medio dejando uno afuera para k = 3, 5 y 9. Para comparar de forma justa, calcula también el ECM de la recta de ML 2 dejando uno afuera (ajusta `np.polyfit` sin el estudiante `i` y predice su nota). ¿Cuál es mejor?

## 🏆 Reto opcional: reconocer dígitos escritos a mano

scikit-learn trae 1797 imágenes de dígitos escritos a mano, de 8 × 8 píxeles. Cada imagen es una fila de 64 números (el brillo de cada píxel), así que **cada imagen es un punto en un espacio de 64 dimensiones**… ¡y la distancia de Pitágoras funciona igual!

```python
from sklearn.datasets import load_digits
digitos = load_digits()
Xd, yd = digitos.data, digitos.target
plt.imshow(digitos.images[0], cmap="gray_r")
```

1. Muestra algunas imágenes con su etiqueta.
2. Usa las primeras 1500 imágenes como tabla del modelo y predice las 297 restantes con `KNeighborsClassifier(n_neighbors=3)`. ¿Qué porcentaje acierta?
3. Muestra las imágenes en las que se equivocó. ¿Se te habrían confundido también a ti?

## 🎬 Cierre

Hoy aprendiste que:
- **Clasificar** es predecir una categoría; **regresión** es predecir un número.
- El algoritmo de los **k vecinos más cercanos** clasifica un dato nuevo por votación entre los k datos más parecidos, usando la **distancia** de Pitágoras.
- Un **k pequeño** se cree hasta los casos raros (frontera irregular); un **k grande** suaviza demasiado.
- Hay que medir la precisión de forma honesta (**dejar uno afuera**) y **normalizar** las variables.
- En scikit-learn todo modelo se usa con `fit` y `predict`.

**Para pensar:** k-NN acierta bastante, pero no nos **explica** su decisión: solo dice «tus vecinos aprobaron». ¿Y si quisiéramos reglas claras, como «si estudias más de 6 horas y duermes más de 7, apruebas»? En la próxima lección construiremos modelos que aprenden reglas así, con forma de árbol: los **árboles de decisión**. 🌳